# 01 — Raw data and ingest

Walkthrough of the OSM snapshot and `build_catalog`. This notebook is the
oracle: the committed fixture `tests/fixtures/catalog.osm`, not a Geofabrik
extract.

**Here:** what a hiking relation looks like, which relations become DayHikes,
ingest stages, progress, resume after a kill.

**Not here:** Catalog tables and maps (`02_catalog.ipynb`), production ranking
(`03_ranking.ipynb`), FeatureVector experiments (`04_rank_feature_lab.ipynb`).

`dayhike` never queries OpenStreetMap at rank time. You ingest a frozen
snapshot once into SQLite. The fixture is a hand-built XML file with a known
answer — not a Croatia dataset.

| OSM relation | Name | Fate |
| --- | --- | --- |
| 1001 | Ridge Walk | **Kept.** ~10 km, `sac_scale=mountain_hiking`, Hut/cafe/parking/water near the Trailhead, `ele` on nodes. A disconnected extra member is skipped. |
| 1002 | Valley Path | **Kept.** ~5 km, no `sac_scale`, no nearby amenities, no `ele`. |
| 1003 | `relation:1003` | **Kept.** No OSM `name`, `ref`, `from`/`to`, or named place nearby. `name_origin=synthetic`. |
| 1004 | Long Traverse | Dropped: ~30 km, outside the 1–20 km band (`dropped_length`). |
| 1005 | Via Dinarica Red | Dropped: `network=iwn` superroute (`dropped_network`). |
| 1006 | Fifty Km Walk | Dropped: ~50 km (`dropped_length`). |
| 1007 | PS-1 | **Kept.** OSM `ref` only (`name_origin=osm_ref`). |

A real Catalog is the same pipeline on Geofabrik `.osm.pbf`. README: *Pulling
a real extract*.


In [ ]:
import tempfile
from pathlib import Path

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

work_dir = Path(tempfile.mkdtemp())
print("fixture", fixture)
print("work_dir", work_dir)


## Raw OSM

Ridge Walk is a `type=route` / `route=hiking` relation. Member ways carry
`highway` and `sac_scale`; nodes may carry `ele`. Unnamed `highway=path` ways
that are not in a hiking relation never become DayHikes.


In [ ]:
text = fixture.read_text()
start = text.index('<relation id="1001"')
end = text.index("</relation>", start) + len("</relation>")
print(text[start:end])
print("---")
print("relations", text.count("<relation "))
print("ways", text.count("<way "))
print("nodes", text.count("<node "))


## Ingest stages

`build_catalog` / `dayhike ingest` **stream** the snapshot. They do not load
the whole file into RAM. A continent PBF uses the same stages.

| Stage | What happens |
| --- | --- |
| `filter` | If `osmium` is on `PATH` and the input is PBF, write a hiking-and-amenity extract. XML skips this. |
| `relations` | Keep `route=hiking` relations and amenity relations. |
| `ways` | Keep member ways of those relations (plus amenity ways). |
| `nodes` | Keep coordinates and `ele` only for those members. |
| `assemble` | Load the work file, then walk each relation's member ways (the main walk, starting near parking). Apply naming and length-band rules; build DayHikes, Trailheads, AmenityFacts. |
| `write` | Write a temp Catalog, then replace `catalog.sqlite`. Also writes the GeometryStore sidecar. |

Progress lines go to the `progress` callback (CLI: stderr). Work lands in
`catalog.sqlite.work` next to the Catalog. Re-running the same inputs resumes
that file. `fresh=True` discards it. The previous Catalog stays until `write`
finishes.


In [ ]:
from dayhike import build_catalog

catalog_path = work_dir / "catalog.sqlite"
messages = []


def show(message: str) -> None:
    messages.append(message)
    print(message)


catalog = build_catalog(fixture, catalog_path, progress=show)
print(catalog.ingest_report)
print("sqlite", catalog.path)
print("geometry", catalog_path.with_name(catalog_path.stem + ".geometry.sqlite").exists())
print("work leftover", catalog_path.with_name(catalog_path.name + ".work").exists())
for hike in catalog.dayhikes:
    print(hike.id, hike.name, round(hike.length_m), hike.name_origin)


## Keep / drop

`IngestReport` is the scoreboard. Unnamed hiking relations are **kept**
(`dropped_unnamed` is 0); `name_origin` records how the Seed label was chosen
(ADR-0017): OSM `name` (`osm_name`); else `ref` with `from – to` or the nearest
place (`osm_ref`); else `from – to` (`osm_from_to`); else `Hike near Place`
(`nearest_place`); else `relation:{id}` (`synthetic`). A
`network:type=node_network` leg that is not a RoundTrip is dropped
(`dropped_node_network`). The fixture has no node-network legs and no
`from`/`to` tags, so those counts are 0 here; `02_catalog.ipynb` shows them on
a real Catalog.


In [ ]:
import os

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")

import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)

report = catalog.ingest_report
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].bar(
    ["kept", "dropped_length", "dropped_network", "dropped_node_network", "dropped_unnamed"],
    [
        report.kept,
        report.dropped_length,
        report.dropped_network,
        report.dropped_node_network,
        report.dropped_unnamed,
    ],
    color=["#2a6", "#c2410c", "#c2410c", "#c2410c", "#999"],
)
axes[0].set_title("IngestReport")
axes[0].tick_params(axis="x", rotation=20)
origins: dict[str, int] = {}
for hike in catalog.dayhikes:
    origins[hike.name_origin] = origins.get(hike.name_origin, 0) + 1
axes[1].bar(list(origins), list(origins.values()), color="#1d4ed8")
axes[1].set_title("name_origin of kept DayHikes")
fig.tight_layout()
show_fig(fig)


## Resume after a kill

If ingest stops mid-run, the Catalog is not replaced. The work file remains.
The next `build_catalog` on the same inputs continues. Completed stages print
`resumed, already complete`.


In [ ]:
resume_path = work_dir / "resume.sqlite"
work_path = resume_path.with_name(resume_path.name + ".work")


def kill_on_assemble(message: str) -> None:
    print(message)
    if message.startswith("assemble:"):
        raise RuntimeError("killed")


try:
    build_catalog(fixture, resume_path, progress=kill_on_assemble)
except RuntimeError as exc:
    print("interrupted", exc)

print("catalog exists", resume_path.exists())
print("work exists", work_path.exists())

resumed = build_catalog(fixture, resume_path, progress=print)
print(resumed.ingest_report)
print("Ridge Walk" in {hike.name for hike in resumed.dayhikes})
print("Valley Path" in {hike.name for hike in resumed.dayhikes})
print("work leftover", work_path.exists())


## Unused objects are skipped

A residential way far from the fixture DayHikes is not a hiking route. Ingest
still keeps Ridge Walk and Valley Path. On a continent extract this is the
same idea: buildings and unrelated roads never become DayHikes and are not
held in RAM.


In [ ]:
noisy = work_dir / "noisy.osm"
extra = "".join(
    f'  <node id="{9000 + i}" lat="46.0" lon="16.0" version="1"/>\n'
    for i in range(20)
)
extra += '''  <way id="9001" version="1">
    <nd ref="9000"/>
    <nd ref="9019"/>
    <tag k="highway" v="residential"/>
  </way>
</osm>
'''
noisy.write_text(fixture.read_text().replace("</osm>", extra))
noisy_catalog = build_catalog(noisy, work_dir / "noisy.sqlite", progress=print)
print(noisy_catalog.ingest_report)
print([hike.name for hike in noisy_catalog.dayhikes])


## CLI

```bash
uv run dayhike ingest tests/fixtures/catalog.osm --catalog data/catalog.sqlite
uv run dayhike ingest data/osm --catalog data/catalog.sqlite
uv run dayhike ingest data/osm --catalog data/catalog.sqlite --fresh
```

Stdout: path and `IngestReport`. Stderr: stage lines. `--list` prints kept
DayHikes. `--fresh` ignores `catalog.sqlite.work`.

If osmium-tool is installed, PBF ingest writes `*.catalog.osm.pbf` next to the
source. DriveTime still needs the unfiltered extract and
`data/valhalla/prepare-roads.sh` — do not feed the hiking extract to Valhalla.

**Next:** [`02_catalog.ipynb`](02_catalog.ipynb) opens the Catalog (and
GeometryStore) and looks at the data.
